# 02 - Data Cleaning & Time Aggregation

The raw data is minute-by-minute. I kept the cleaning rules simple and made the aggregation coverage visible.

In [ ]:
import pandas as pd

file = "../01_Data/raw/household_power_consumption.csv"
df = pd.read_csv(file, na_values="?")
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df = df.set_index("datetime")

In [ ]:
cols = ["Global_active_power", "Global_reactive_power", "Voltage",
        "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

# Keep a separate copy for small-window analysis.
filled = df.copy()
filled[cols] = filled[cols].interpolate(limit=60, limit_direction="both")

In [ ]:
daily = df.resample("D").agg(
    daily_kwh=("Global_active_power", lambda s: s.sum(min_count=1)/60),
    avg_voltage=("Voltage", "mean"),
    avg_intensity=("Global_intensity", "mean"),
    records=("Global_active_power", "count")
)

daily["coverage_pct"] = daily["records"] / 1440 * 100
daily = daily[daily["coverage_pct"] >= 90]
daily.to_csv("../01_Data/processed/daily_energy.csv")
print(daily.shape)

In [ ]:
hourly = df.resample("h").agg(
    energy_kwh=("Global_active_power", lambda s: s.sum(min_count=1)/60),
    avg_voltage=("Voltage", "mean"),
    avg_intensity=("Global_intensity", "mean"),
    records=("Global_active_power", "count")
)
hourly["coverage_pct"] = hourly["records"] / 60 * 100
hourly = hourly[hourly["coverage_pct"] >= 90]
hourly.to_csv("../01_Data/processed/hourly_energy.csv")